# Best Practices for Building a Modern App with Vector Search

This notebook demonstrates how to build a modern LLM-powered application using:

- **Jina Embeddings v5** via Elastic Inference Service (EIS) GPU-accelerated multilingual embeddings
- **Elasticsearch 9.3+** for vector storage and semantic search
- **Agent Builder** for creating AI agents that can query your data

## What You'll Learn

1. Setting up inference endpoints for embeddings
2. Creating optimized indices for vector search
3. Ingesting data with automatic embedding generation
4. Performing semantic searches
5. Building an AI agent with Agent Builder

## Setup and Configuration

In [ ]:
# Install required packages
!pip install elasticsearch requests dotenv -q

Create a `.env` file in the project root with your Elasticsearch and Kibana credentials:

```
ELASTICSEARCH_URL=https://your-cluster-url
ELASTICSEARCH_API_KEY=your-api-key
KIBANA_URL=https://your-kibana-url
```

In [ ]:
import os
import json

import requests
from dotenv import load_dotenv
from elasticsearch import Elasticsearch

load_dotenv()

# Elasticsearch configuration
ELASTICSEARCH_URL = os.getenv("ELASTICSEARCH_URL")
ELASTIC_API_KEY = os.getenv("ELASTICSEARCH_API_KEY")
KIBANA_URL = os.getenv("KIBANA_URL")

# Initialize Elasticsearch client
es_client = Elasticsearch(ELASTICSEARCH_URL, api_key=ELASTIC_API_KEY)

## Tip 1: Use Managed Inference Instead of External Embedding APIs

We'll use **Jina Embeddings v5** through Elastic Inference Service (EIS).

In [ ]:
INFERENCE_ENDPOINT_ID = ".jina-embeddings-v5-text-small"

# Jina Embeddings v5 comes with a preconfigured endpoint in EIS.
# No need to create a custom inference endpoint; just reference the preconfigured ID.
print(f"Using preconfigured inference endpoint: {INFERENCE_ENDPOINT_ID}")

## Tip 2: Design Hybrid-Ready Indices from Day One

In [ ]:
INDEX_NAME = "tech-articles"

index_mappings = {
    "mappings": {
        "properties": {
            "title": {
                "type": "text",
                "copy_to": "semantic_field",
                "fields": {"keyword": {"type": "keyword"}},
            },
            "content": {"type": "text", "copy_to": "semantic_field"},
            "category": {"type": "keyword"},
            "published_date": {"type": "date"},
            "semantic_field": {
                "type": "semantic_text",
                "inference_id": INFERENCE_ENDPOINT_ID,
            },
        }
    }
}

if not es_client.indices.exists(index=INDEX_NAME):
    response = es_client.indices.create(index=INDEX_NAME, body=index_mappings)
    print(f"Created index: {INDEX_NAME}")
else:
    print(f"Index '{INDEX_NAME}' already exists, skipping creation")

## Tip 3: Use Bulk Operations for Scalable Ingestion

In [ ]:
from elasticsearch import helpers


def build_data(json_file, index_name):
    """Generator function to yield documents for bulk indexing."""
    with open(json_file, "r") as f:
        data = json.load(f)

    for doc in data:
        yield {"_index": index_name, "_source": doc}


# Bulk index the documents from JSON file
try:
    success, failed = helpers.bulk(
        es_client,
        build_data("dataset.json", INDEX_NAME),
    )
    print(f"{success} documents indexed successfully")

    if failed:
        print(f"Errors: {failed}")
except Exception as e:
    print(f"Error: {str(e)}")

In [ ]:
# Check document count
count = es_client.count(index=INDEX_NAME)
print(f"Total documents in index: {count.body['count']}")

## Tip 4: Use a Hybrid Search Strategy

### Reciprocal Rank Fusion (RRF)

In [ ]:
def hybrid_search_rrf(query: str, size: int = 3):
    """Hybrid search using RRF."""

    response = es_client.search(
        index=INDEX_NAME,
        body={
            "retriever": {
                "rrf": {
                    "retrievers": [
                        {
                            "standard": {
                                "query": {
                                    "multi_match": {
                                        "query": query,
                                        "fields": ["title^2", "content"],
                                    }
                                }
                            }
                        },
                        {
                            "standard": {
                                "query": {"match": {"semantic_field": {"query": query}}}
                            }
                        },
                    ],
                    "rank_window_size": 50,
                    "rank_constant": 20,
                }
            },
            "size": size,
            "_source": ["title", "category"],
        },
    )

    return response

In [ ]:
response = hybrid_search_rrf(
    "What are the best practices for semantic search in Elasticsearch?"
)

print("Hybrid Search Results:")
for hit in response.body["hits"]["hits"]:
    print(f"Title: {hit['_source']['title']}, Category: {hit['_source']['category']}")

### Reranking with Jina Reranker v3

#### Create inference endpoint for Jina Reranker v3

In [ ]:
RERANKER_INFERENCE_ID = "jina-reranker-v3-endpoint"

reranker_config = {
    "service": "elastic",
    "service_settings": {"model_id": "jina-reranker-v3"},
}

try:
    es_client.inference.put(
        inference_id=RERANKER_INFERENCE_ID,
        task_type="rerank",
        body=reranker_config,
    )

    print(f"Created reranker endpoint: {RERANKER_INFERENCE_ID}")
except Exception as e:
    if "already_exists" in str(e).lower():
        print(f"Reranker endpoint already exists: {RERANKER_INFERENCE_ID}")
    else:
        raise e

In [ ]:
def hybrid_search_with_reranking(query: str, size: int = 3):
    """Hybrid search: RRF retrieval followed by Jina Reranker v3."""
    response = es_client.search(
        index=INDEX_NAME,
        body={
            "retriever": {
                "text_similarity_reranker": {
                    "retriever": {
                        "rrf": {
                            "retrievers": [
                                {
                                    "standard": {
                                        "query": {
                                            "multi_match": {
                                                "query": query,
                                                "fields": ["title^2", "content"],
                                            }
                                        }
                                    }
                                },
                                {
                                    "standard": {
                                        "query": {
                                            "match": {
                                                "semantic_field": {"query": query}
                                            }
                                        }
                                    }
                                },
                            ],
                            "rank_window_size": 50,
                            "rank_constant": 20,
                        }
                    },
                    "field": "semantic_field",
                    "inference_id": RERANKER_INFERENCE_ID,
                    "inference_text": query,
                    "rank_window_size": 25,
                }
            },
            "size": size,
            "_source": ["title", "category"],
        },
    )
    return response

In [ ]:
response = hybrid_search_with_reranking(
    "What are the best practices for semantic search in Elasticsearch?"
)

print("Hybrid Search + Reranking Results:")
for hit in response.body["hits"]["hits"]:
    print(f"Title: {hit['_source']['title']}, Category: {hit['_source']['category']}")

## Tip 5: Add AI Reasoning with Agent Builder

In [ ]:
headers = {
    "kbn-xsrf": "true",
    "Authorization": f"ApiKey {ELASTIC_API_KEY}",
    "Content-Type": "application/json",
}

# Create a custom ES|QL tool that returns articles filtered by publish date.
esql_tool_payload = {
    "id": "tech-articles-recent",
    "type": "esql",
    "description": (
        "Returns recent articles from the knowledge base filtered by publish date. "
        "Use this when the user asks about recent content, articles from a specific "
        "time period, or wants to browse by date."
    ),
    "tags": ["analytics", "tech-articles"],
    "configuration": {
        "query": (
            "FROM tech-articles "
            "| WHERE published_date >= ?startDate "
            "| KEEP title, category, published_date "
            "| SORT published_date DESC "
            "| LIMIT ?limit"
        ),
        "params": {
            "startDate": {
                "type": "date",
                "description": "Start date for filtering articles (ISO 8601, e.g. 2024-01-01)",
            },
            "limit": {
                "type": "integer",
                "description": "Maximum number of articles to return",
            },
        },
    },
}

response = requests.post(
    f"{KIBANA_URL}/api/agent_builder/tools",
    headers=headers,
    json=esql_tool_payload,
)

if response.status_code == 200:
    print(f"Created custom ES|QL tool: {esql_tool_payload['id']}")
else:
    print(f"Error creating tool: {response.text}")

agent_payload = {
    "id": "tech-articles-assistant",
    "name": "Tech Articles Assistant",
    "description": "An AI assistant that helps users find information about technology topics from our knowledge base.",
    "configuration": {
        # Uses Elastic Managed LLM by default. No connector_id needed.
        "tools": [
            {
                "tool_ids": [
                    "platform.core.search",
                    "platform.core.execute_esql",
                    "tech-articles-recent",
                ]
            }
        ],
        "instructions": f"""You are a helpful assistant that answers questions about technology topics.

Use the search tool to find relevant articles from the '{INDEX_NAME}' index.
Use the tech-articles-recent tool when the user asks about recent or time-filtered content.
When searching, prefer semantic search for natural language questions.
Always cite the article titles when providing information.
If you cannot find relevant information, say so clearly.""",
    },
}

response = requests.post(
    f"{KIBANA_URL}/api/agent_builder/agents",
    headers=headers,
    json=agent_payload,
    verify=True,
)

if response.status_code == 200:
    agent_data = response.json()
    agent_id = agent_data.get("id")
    print(f"Created agent: {agent_id}")
else:
    print(f"Error creating agent: {response.text}")
    agent_id = None

In [ ]:
def chat_with_agent(agent_id: str, message: str):
    """Send a message to the agent and get a response."""
    chat_payload = {"input": message, "agent_id": agent_id}

    response = requests.post(
        f"{KIBANA_URL}/api/agent_builder/converse",
        headers=headers,
        json=chat_payload,
        verify=True,
    )

    if response.status_code == 200:
        return response.json()
    else:
        return {"error": response.text, "status_code": response.status_code}

In [ ]:
# Example conversation
if agent_id:
    result = chat_with_agent(
        agent_id, "What are the best practices for building RAG applications?"
    )
    print(json.dumps(result, indent=2))

## Tip 6: Automate with Elastic Workflows

**Elastic Workflows** is an automation engine built into Elasticsearch that orchestrates multi-step processes using YAML.

> **Note:** Workflows is in technical preview as of Elasticsearch 9.3. To enable the Workflows UI,
> go to Kibana: `Stack Management > Advanced Settings`, and set `workflows:ui:enabled` to `true`.
> Paste the YAML below into the editor to create this workflow.

![image.png](assets/workflow-driagram.png)

In [ ]:
"""
name: Hacker News Digest
description: >
  Fetches the latest top stories from the Hacker News public API, indexes them
  into Elasticsearch with semantic embeddings, then asks the AI agent to
  summarize the key themes from the freshly ingested content.
enabled: true
tags: ["ingestion", "hacker-news", "agent", "demo"]

consts:
  indexName: tech-articles
  hnApiBase: "https://hacker-news.firebaseio.com/v0"
  agentId: tech-articles-assistant

triggers:
  - type: manual

steps:
  # Step 1: Fetch top story IDs from the Hacker News public API (no auth required)
  - name: fetch_top_stories
    type: http
    with:
      url: "{{ consts.hnApiBase }}/topstories.json"
      method: GET

  # Step 2: For each story ID, fetch details and index into Elasticsearch.
  - name: process_stories
    type: foreach
    foreach: "${{ steps.fetch_top_stories.output.data | slice: 0, 5 }}"
    steps:
      - name: fetch_story_detail
        type: http
        with:
          url: "{{ consts.hnApiBase }}/item/{{ foreach.item }}.json"
          method: GET

      - name: index_story
        type: elasticsearch.request
        with:
          method: POST
          path: "/{{ consts.indexName }}/_doc"
          body:
            title: "{{ steps.fetch_story_detail.output.data.title }}"
            content: "{{ steps.fetch_story_detail.output.data.text | default: steps.fetch_story_detail.output.data.title }}"
            category: "hacker-news"
            url: "{{ steps.fetch_story_detail.output.data.url }}"

  # Step 3: Ask the agent to summarize the freshly indexed stories.
  - name: ask_agent
    type: ai.agent
    with:
      agent_id: "{{ consts.agentId }}"
      message: "What are the main themes and topics from the latest Hacker News stories?"

  - name: log_summary
    type: console
    with:
      message: "{{ steps.ask_agent.output }}"
"""

### Results of Workflow Creation

![image.png](assets/workflow-result.png)

In [ ]:
es_client.indices.delete(index=INDEX_NAME)